In [ ]:
# every process contains atleast one thread, the main thread. This thread is what begins execution of the code. If there's code written for creating more threads, then we get multiple threads being concurrent, but not parallel (atleast for purely CPU bound tasks), thanks to the GIL. that's a multi-threaded program.

# NOTE: I made the case only for cpu-bound tasks because had it been IO-bound, then the GIL is explicitly released (Cpython is written this way) for every IO call... In fact it is even released periodically for cpu bound tasks as well (every 5ms or so) to give every thread a chance to execute and reduce chance for starvation.

# NOTE 2: All io calls are sys calls (calls to os to manipulate hardware), but not all sys calls are io calls (malloc(), getpid(), clone()/ fork()). Cpython has 2 macros wrapped around most sys calls meant for IO, the top macro releases gil, the bottom macro reacquires it.

# e.g. 

# Py_BEGIN_ALLOW_THREADS   // → releases the GIL, save thread state
# result = read(fd, buffer, size);  // actual OS syscall
# Py_END_ALLOW_THREADS     // → reacquires the GIL, reload thread state

import os
import threading

def main():
    print(f'Python process running with process id: {os.getpid()}')

    total_threads = threading.active_count()
    thread_name = threading.current_thread().name
    print(f'Python is currently running {total_threads} thread(s)')
    print(f'The current thread is {thread_name}')

if __name__ == '__main__':
    main()

### simple multithreaded program

In [ ]:
import os
import threading

def hello_from_thread():
    print(f'Hello from thread {threading.current_thread()}!')
    thread_name = threading.current_thread().name
    print(f'The current thread is {thread_name}')

def main():
    hello_thread = threading.Thread(target=hello_from_thread)
    hello_thread.start()
    total_threads = threading.active_count()
    thread_name = threading.current_thread().name
    print(f'Python is currently running {total_threads} thread(s)')
    print(f'The current thread is {thread_name}')
    hello_thread.join()

if __name__ == '__main__':
    main()

### simple multiprocess program

In [ ]:
import multiprocessing
import os

def hello_from_process():
    print(f'Hello from child process {os.getpid()}!')

if __name__ == '__main__':
    hello_process = multiprocessing.Process(target=hello_from_process)
    hello_process.start()
    print(f'Hello from parent process {os.getpid()}')
    hello_process.join()

### Understanding the GIL

GIL prevents your threads from running code parallely.. 

In [ ]:
# finding the nth fibonacci number, single-threaded

import time

def print_fib(number: int) -> None:
    def fib(n: int) -> int:
        if n == 1:
            return 0
        elif n == 2:
            return 1
        else:
            return fib(n - 1) + fib(n - 2)
            
    print(f'fib({number}) is {fib(number)}')
    
def fibs_no_threading():
    print_fib(40)
    print_fib(41)

if __name__ == '__main__':
    start = time.time()
    fibs_no_threading()
    end = time.time()
    print(f'Completed in {end - start:.4f} seconds.') # ~22.3s

In [ ]:
# with multithreading... 

import threading
import time

def print_fib(number: int) -> None:
    def fib(n: int) -> int:
        if n == 1:
            return 0
        elif n == 2:
            return 1
        else:
            return fib(n - 1) + fib(n - 2)
            
    print(f'fib({number}) is {fib(number)}')
    
def fibs_with_threads():
    fortieth_thread = threading.Thread(target=print_fib, args=(40,))
    forty_first_thread = threading.Thread(target=print_fib, args=(41,))
    fortieth_thread.start()
    forty_first_thread.start()
    fortieth_thread.join()
    forty_first_thread.join()

if __name__ == '__main__':
    start_threads = time.time()
    fibs_with_threads()
    end_threads = time.time()
    print(f'Threads took {end_threads - start_threads:.4f} seconds.') # 23.7s, even slower than single-threaded! This is because of GIL (and to a lesser extent, overhead of creating threads.)


# NOTE: this makes threads pretty useless for CPU-bound tasks, where you're trying to stack computations parallely... This simply isn't possible with threads due to the GIL.

Multi-threaded programs are still great for IO-tasks, since the GIL is released for IO-tasks in threads, allowing for some meaningful concurrency.

In [ ]:
# reaching out to a server 5 times, synchronously

import time
import requests

def read_example() -> None:
    response = requests.get('https://www.example.com')
    print(response.status_code)

if __name__ == '__main__':
    sync_start = time.time()
    for i in range(5):
        read_example()
    sync_end = time.time()
    print(f'Running synchronously took {sync_end - sync_start:.4f} seconds.') # ~0.47s

In [ ]:
# reaching out to a server 5 times, in multi-threaded manner

import time
import requests
import threading

THREAD_COUNT = 5

def read_example() -> None:
    response = requests.get('https://www.example.com')
    print(response.status_code)

if __name__ == '__main__':
    sync_start = time.time()
    threads = []
    for i in range(THREAD_COUNT):
        t = threading.Thread(target=read_example)
        t.start()
        threads.append(t)

    for t in threads:
        t.join()

    sync_end = time.time()
    print(f'Running multithreaded took {sync_end - sync_start:.4f} seconds.') # 0.17s

Asyncio takes advantage of the fact that IO operations are non-blocking, and that the GIL is released when an IO operation is performed. 

Thanks to this, we don't even need multiple threads to achieve a high level of concurrency when it comes to IO operations.. a single thread, with proper asynchronous code written can do the job for us.

Asyncio achieves this by using **non-blocking sockets**

### socket

think of it like a literal socket you plug in your wire to. Author used the mailboxes wala analogy.

sockets can receive bytes, or send bytes.

you use these for IO operations..

However, by default, sockets are blocking, i.e. once an IO request has been made, you just wait there until you have some information to read from socket.
This need not be the case. You can turn the sockets into non-blocking mode, and eliminate this problem...

Now, for every IO request, you register it on the socket, and then you can keep executing other python code.. Until the result is ready, you (the event loop) keep polling the OS's **event notification system**, asking whether the task is done. Once it affirms, you collect your result and unblock whatever task was waiting on it.

Besides leaving you free to execute other python code, this is awesome for another reason, that you can also stack these non-blocking sockets together to set in motion multiple IO requests, cutting down on waiting time...

`Non-blocking socket → registered with OS → event loop keeps checking "is it ready?" → when yes, resumes the coroutine that was waiting on it via await.`